# DropItRight — per-extractor debug notebook

Runs every extractor individually on ONE song (whole audio + every segment) and prints raw output, so you can sanity-check each stage in isolation rather than only seeing the final fused report.

Run cells top to bottom. Cell 1 sets `AUDIO_PATH` — change it and re-run from Cell 2 down to inspect a different song.

Order mirrors `process_song.py`: beat tracking -> segmentation -> global features (tonic, raga, MERT, lyrics) -> per-segment features (CAE, melodysim, lyric slice) for every segment at every scale.

In [ ]:
# Cell 1 -- config
import os, sys, logging
logging.basicConfig(level=logging.INFO)

# Point this at whichever DropItRight checkout you're running from, if the
# notebook isn't already sitting in the project root.
PROJECT_ROOT = os.path.abspath(".")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

AUDIO_PATH = "tests/audio/QRY_003_Krazzy4_Remix.wav"  # <-- change per song
DEVICE = "cuda"  # or "cpu"
TMP_DIR = "tmp_debug_segments"
os.makedirs(TMP_DIR, exist_ok=True)

import numpy as np
import soundfile as sf

info = sf.info(AUDIO_PATH)
print(f"audio: {AUDIO_PATH}")
print(f"duration: {info.duration:.2f}s | sr: {info.samplerate} | channels: {info.channels}")

In [ ]:
# Cell 1b -- listen to the whole song
from IPython.display import Audio
Audio(AUDIO_PATH)

## 1. Beat / downbeat tracking (`beat_tracking.track_beats`)

In [ ]:
# Cell 2 -- beat tracking
import beat_tracking

beat_result = beat_tracking.track_beats(AUDIO_PATH)
beat_times, downbeat_start, rhythm, bpm = beat_result.as_tuple()

print(f"source: {beat_result.source}")
print(f"bpm: {bpm}")
print(f"rhythm (beats/bar): {rhythm}")
print(f"downbeat_start: {downbeat_start:.3f}s")
print(f"n beats: {len(beat_times)}")
print(f"first 10 beat times: {np.asarray(beat_times[:10]).round(3)}")

## 2. Phrase-aligned segmentation (`segmentation.segment_phrases`)

In [ ]:
# Cell 3 -- segmentation
import segmentation

try:
    segments_by_scale = segmentation.segment_phrases(beat_times, downbeat_start, rhythm)
except Exception as exc:
    print(f"phrase segmentation failed ({exc}) -- falling back to fixed windows")
    segments_by_scale = segmentation.segment_phrases_fixed_window(info.duration)

for duration_class, segs in segments_by_scale.items():
    print(f"\n-- duration_class={duration_class} ({len(segs)} segments) --")
    for s in segs:
        print(f"  [{s.start:7.2f}s -> {s.end:7.2f}s]  bars {s.bar_start_idx}->{s.bar_end_idx}")

## 3. Global (whole-song) features

In [ ]:
# Cell 4 -- tonic identification
import indian_features

tonic_hz = None
try:
    tonic_hz = indian_features.extract_tonic(AUDIO_PATH)
    print(f"tonic_hz: {tonic_hz:.3f} Hz")
except Exception as exc:
    print(f"tonic extraction FAILED: {exc}")

In [ ]:
# Cell 5 -- raga recognition (DEEPSRGM)
raga_info = {"raga": None, "confidence": None}
try:
    raga_info = indian_features.extract_raga(audio_path=AUDIO_PATH)
    print(f"raga: {raga_info['raga']}")
    print(f"confidence: {raga_info['confidence']}")
except Exception as exc:
    print(f"raga recognition FAILED: {exc}")

In [ ]:
# Cell 6 -- MERT whole-song embedding
import global_embeddings

mert_embedding = None
try:
    mert_embedding = global_embeddings.extract_mert_embedding(AUDIO_PATH, device=DEVICE)
    print(f"shape: {mert_embedding.shape}")
    print(f"dtype: {mert_embedding.dtype}")
    print(f"norm: {np.linalg.norm(mert_embedding):.4f}")
    print(f"first 10 dims: {mert_embedding[:10].round(4)}")
    print(f"stats: min={mert_embedding.min():.4f} max={mert_embedding.max():.4f} mean={mert_embedding.mean():.4f}")
except Exception as exc:
    print(f"MERT embedding FAILED: {exc}")

In [ ]:
# Cell 7 -- lyrics transcription (Indic ASR)
global_lyrics = None
try:
    global_lyrics = global_embeddings.extract_lyrics(AUDIO_PATH, device=DEVICE)
    print(f"full text:\n{global_lyrics['text']}\n")
    print(f"n timestamped chunks: {len(global_lyrics['chunks'])}")
    for c in global_lyrics["chunks"][:15]:
        ts = c.get("timestamp", (None, None))
        print(f"  [{ts[0]}, {ts[1]}]  {c.get('text')!r}")
except Exception as exc:
    print(f"lyrics transcription FAILED: {exc}")

## 4. Per-segment features

Renders every segment at every scale to a temp wav, then runs CAE-Carnatic, melodysim, and the lyric-slice cut on each. Same order/logic as `process_song.py` step 4.

In [ ]:
# Cell 8 -- render + extract per-segment features for ALL segments/scales
import librosa
import melodysim_embeddings
from process_song import _render_segment_wav

segment_results = []  # list of dicts, one per segment, for inspection below

for duration_class, segs in segments_by_scale.items():
    for seg in segs:
        seg_wav = os.path.join(TMP_DIR, f"debug_{duration_class}_{seg.start:.2f}.wav")
        record = {
            "duration_class": duration_class,
            "start": seg.start,
            "end": seg.end,
        }
        try:
            _render_segment_wav(AUDIO_PATH, seg.start, seg.end, seg_wav)

            # CAE-Carnatic embedding
            try:
                cae_vec = indian_features.extract_cae_embedding(seg_wav, device=DEVICE)
                record["cae_embedding"] = cae_vec
                record["cae_error"] = None
            except Exception as exc:
                record["cae_embedding"] = None
                record["cae_error"] = str(exc)

            # melodysim embedding (expected to fail until wired up)
            try:
                record["melodysim_embedding"] = melodysim_embeddings.extract_melodysim_embedding(
                    seg_wav, device=DEVICE
                )
                record["melodysim_error"] = None
            except NotImplementedError as exc:
                record["melodysim_embedding"] = None
                record["melodysim_error"] = "not wired up (NotImplementedError)"
            except Exception as exc:
                record["melodysim_embedding"] = None
                record["melodysim_error"] = str(exc)

            # lyric slice from the already-computed global transcript
            record["lyrics_slice"] = (
                global_embeddings.slice_lyrics(global_lyrics, seg.start, seg.end)
                if global_lyrics is not None else None
            )
        finally:
            if os.path.exists(seg_wav):
                os.remove(seg_wav)

        segment_results.append(record)

print(f"processed {len(segment_results)} segments across scales: {list(segments_by_scale.keys())}")

In [ ]:
# Cell 9b -- listen to a specific segment (edit SEG_INDEX / pick from segment_results_sorted)
import librosa
from IPython.display import Audio

SEG_INDEX = 0  # <-- index into segment_results_sorted
seg = segment_results_sorted[SEG_INDEX]
print(f"duration_class={seg['duration_class']}  [{seg['start']:.2f}s -> {seg['end']:.2f}s]")

y, sr = librosa.load(AUDIO_PATH, sr=None, offset=seg["start"], duration=seg["end"] - seg["start"])
Audio(y, rate=sr)

In [ ]:
# Cell 9 -- print per-segment results, grouped by duration_class
from itertools import groupby

segment_results_sorted = sorted(segment_results, key=lambda r: (str(r["duration_class"]), r["start"]))

for duration_class, group in groupby(segment_results_sorted, key=lambda r: r["duration_class"]):
    group = list(group)
    print(f"\n=== duration_class={duration_class} ({len(group)} segments) ===")
    for r in group:
        print(f"\n  [{r['start']:.2f}s -> {r['end']:.2f}s]")
        if r["cae_embedding"] is not None:
            v = r["cae_embedding"]
            print(f"    cae_embedding: shape={v.shape} norm={np.linalg.norm(v):.4f} first5={v[:5].round(4)}")
        else:
            print(f"    cae_embedding: FAILED ({r['cae_error']})")
        if r["melodysim_embedding"] is not None:
            v = r["melodysim_embedding"]
            print(f"    melodysim_embedding: {v}")
        else:
            print(f"    melodysim_embedding: n/a ({r['melodysim_error']})")
        print(f"    lyrics_slice: {r['lyrics_slice']!r}")

## 5. Sanity checks / summary

Quick pass/fail table across every extractor so you can see at a glance what's live vs. stubbed/broken for this song.

In [ ]:
# Cell 10 -- summary table
n_cae_ok = sum(1 for r in segment_results if r["cae_embedding"] is not None)
n_melodysim_ok = sum(1 for r in segment_results if r["melodysim_embedding"] is not None)
n_lyrics_ok = sum(1 for r in segment_results if r["lyrics_slice"])

tonic_status = f"ok, {round(tonic_hz, 2)} Hz" if tonic_hz else "FAILED"
raga_status = f"ok, {raga_info['raga']}" if raga_info["raga"] else "FAILED / None"
mert_status = f"ok, dim={mert_embedding.shape[0]}" if mert_embedding is not None else "FAILED"
lyrics_status = f"ok, {len(global_lyrics['chunks'])} chunks" if global_lyrics else "FAILED"

print(f"{'extractor':<28} {'status'}")
print(f"{'-'*28} {'-'*40}")
print(f"{'beat_tracking':<28} ok (source={beat_result.source}, bpm={bpm})")
print(f"{'segmentation':<28} ok ({len(segment_results)} total segments across {len(segments_by_scale)} scales)")
print(f"{'tonic':<28} {tonic_status}")
print(f"{'raga (DEEPSRGM)':<28} {raga_status}")
print(f"{'MERT embedding':<28} {mert_status}")
print(f"{'lyrics (global ASR)':<28} {lyrics_status}")
print(f"{'cae per-segment':<28} {n_cae_ok}/{len(segment_results)} segments ok")
print(f"{'melodysim per-segment':<28} {n_melodysim_ok}/{len(segment_results)} segments ok (expected 0 until wired up)")
print(f"{'lyric slices':<28} {n_lyrics_ok}/{len(segment_results)} segments have non-empty text")